In [ ]:
import os, subprocess
os.makedirs("build", exist_ok=True)

# Day 1 Lab: Welcome to Hardware Thinking

> **Week 1, Session 1** · Accelerated HDL for Digital System Design · UCF ECE

## Overview

| | |
|---|---|
| **Duration** | ~2 hours |
| **Prerequisites** | Pre-class video (40 min): HDL vs. software, synthesis vs. simulation, module anatomy, digital logic refresher |
| **Deliverable** | Buttons-to-LEDs with at least one logic modification, programmed on Go Board |
| **Tools** | Yosys, nextpnr-ice40, icepack, iceprog, Icarus Verilog, GTKWave |

## Learning Objectives

| SLO | Description |
|-----|-------------|
| 1.1 | Explain why `assign` statements execute concurrently, not sequentially |
| 1.3 | Write a syntactically correct Verilog module with ANSI-style ports |
| 1.4 | Execute the full iCE40 toolchain: Yosys → nextpnr → icepack → iceprog |
| 1.5 | Use a `.pcf` file to map HDL signals to physical Go Board pins |
| 1.6 | Predict and verify the behavior of active-low LEDs and buttons |

## Exercises

### Setup Verification (15 min)

Verify the full toolchain is installed and the Go Board is connected. Run the checklist in `starter/setup_check.sh` or verify manually:

In [ ]:
!yosys --version

In [ ]:
!nextpnr-ice40 --version

In [ ]:
!icepack

In [ ]:
!iceprog

In [ ]:
!iverilog -V

In [ ]:
!gtkwave --version

If any tool is missing, **stop and fix it now** — don't proceed with a broken toolchain.

---

### Exercise 1: LED On — The Simplest Possible Design (20 min)

**Goal:** Write, synthesize, and program the absolute minimum Verilog design. Confirm the full toolchain works end-to-end.

- Open `starter/w1d1_ex1_led_on.v` — the module is complete, just review it
- Build and program: `make ex1`
- **Verify:** LED1 on the Go Board should be lit
- **Reflection:** What did Yosys actually synthesize here? Run `make ex1_stat` to check LUT usage

---

#### 📝 Exercise 1 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex1_led_on.v
// =============================================================================
// Exercise 1: LED On — The Simplest Possible Design
// Day 1 · Welcome to Hardware Thinking
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Goal: Drive LED1 permanently on. Confirm the full toolchain works.
//
// Go Board: LEDs are active low (0 = on, 1 = off)
// =============================================================================

module led_on (
    output wire o_led1
);

    // Drive LED1 on: active low means assign 0 to turn on
    assign o_led1 = 1'b0;

endmodule

### Exercise 2: Buttons to LEDs — Wires in Hardware (25 min)

**Goal:** Use `assign` to create combinational connections between inputs and outputs.

- Open `starter/w1d1_ex2_buttons_to_leds.v` — direct mapping provided
- Build and program: `make ex2`
- **Verify:** Each button controls its corresponding LED
- **Quick check:** Are any LUTs used? Why or why not?

---

#### 📝 Exercise 2 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex2_buttons_to_leds.v
// =============================================================================
// Exercise 2: Buttons to LEDs — Wires in Hardware
// Day 1 · Welcome to Hardware Thinking
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Goal: Wire each button directly to its corresponding LED using assign.
//
// Go Board: Buttons and LEDs are both active low.
//   Button pressed = 0, LED on = 0
//   Direct connection gives intuitive behavior (press → light)
// =============================================================================

module buttons_to_leds (
    input  wire i_switch1,
    input  wire i_switch2,
    input  wire i_switch3,
    input  wire i_switch4,
    output wire o_led1,
    output wire o_led2,
    output wire o_led3,
    output wire o_led4
);

    // Direct connection: each button drives its LED
    // Both are active low, so this gives intuitive behavior
    assign o_led1 = i_switch1;
    assign o_led2 = i_switch2;
    assign o_led3 = i_switch3;
    assign o_led4 = i_switch4;

endmodule

### Exercise 3: Logic Between Buttons and LEDs (30 min)

**Goal:** Implement concurrent combinational logic with multiple independent `assign` statements.

- Open `starter/w1d1_ex3_button_logic.v` — fill in the `TODO` assignments
- Predict the truth tables **on paper first**, then verify on hardware
- Build and program: `make ex3`
- **Discussion:** All four `assign` statements are active simultaneously

| sw1 | sw2 | LED1 (AND) | LED3 (XOR) | LED4 (NOT) |
|-----|-----|-----------|-----------|-----------|
| released | released | ? | ? | ? |
| released | pressed | ? | ? | ? |
| pressed | released | ? | ? | ? |
| pressed | pressed | ? | ? | ? |

---

#### 📝 Exercise 3 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex3_button_logic.v
// =============================================================================
// Exercise 3: Logic Between Buttons and LEDs
// Day 1 · Welcome to Hardware Thinking
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Goal: Implement combinational logic between buttons and LEDs.
//       All assign statements execute CONCURRENTLY — this is hardware!
//
// Remember: Active low on Go Board
//   Button pressed = 0, LED on = 0
//   Think through the truth tables carefully with active-low signals.
// =============================================================================

module button_logic (
    input  wire i_switch1,
    input  wire i_switch2,
    input  wire i_switch3,
    input  wire i_switch4,
    output wire o_led1,
    output wire o_led2,
    output wire o_led3,
    output wire o_led4
);

    // LED1: ON when BOTH switch1 AND switch2 are pressed
    //
    // Truth table (active low):
    //   sw1  sw2  | led1 (0=on)
    //    0    0   |  0    (both pressed -> LED on)
    //    0    1   |  1    (only sw1 -> LED off)
    //    1    0   |  1    (only sw2 -> LED off)
    //    1    1   |  1    (neither -> LED off)
    //
    // TODO: What single gate operation on active-low signals
    //       gives us AND-of-pressed behavior?
    //       Hint: OR of active-low = AND of pressed
    assign o_led1 = 1'b1;  // TODO: replace with correct logic

    // LED2: ON when EITHER switch3 OR switch4 is pressed
    //
    // TODO: What gate gives us OR-of-pressed with active-low signals?
    //       Hint: AND of active-low = OR of pressed
    assign o_led2 = 1'b1;  // TODO: replace with correct logic

    // LED3: ON when exactly ONE of switch1/switch2 is pressed (XOR)
    //
    // TODO: XOR of active-low inputs, then invert for active-low output
    assign o_led3 = 1'b1;  // TODO: replace with correct logic

    // LED4: INVERTED behavior of switch1 (LED on when NOT pressed)
    //
    // TODO: Simply invert switch1
    assign o_led4 = 1'b1;  // TODO: replace with correct logic

endmodule

### Exercise 4: Active-Low Thinking (20 min)

**Goal:** Develop a clean pattern for handling active-low signals.

- Open `starter/w1d1_ex4_active_low_clean.v` — fill in the `TODO` sections
- The pattern: invert active-low inputs at the boundary, work in active-high internally, invert outputs at the boundary
- Build and program: `make ex4`
- **Compare:** Does this produce more or fewer LUTs than Exercise 3? Run `make ex4_stat`

---

#### 📝 Exercise 4 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex4_active_low_clean.v
// =============================================================================
// Exercise 4: Active-Low Clean Pattern
// Day 1 · Welcome to Hardware Thinking
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Goal: Develop the "invert at boundaries" pattern for active-low signals.
//
// Pattern:
//   1. Invert active-low inputs at the top (boundary) -> active-high wires
//   2. Write all internal logic in active-high (natural, readable)
//   3. Invert outputs at the bottom (boundary) -> active-low for LEDs
//
// This keeps your logic clean and readable — the messiness is contained.
// =============================================================================

module active_low_clean (
    input  wire i_switch1,
    input  wire i_switch2,
    input  wire i_switch3,
    input  wire i_switch4,
    output wire o_led1,
    output wire o_led2,
    output wire o_led3,
    output wire o_led4
);

    // ---- Step 1: Invert active-low inputs at the boundary ----
    // TODO: Create active-high wires from the active-low switches
    wire w_btn1, w_btn2, w_btn3, w_btn4;
    // assign w_btn1 = ???;
    // assign w_btn2 = ???;
    // assign w_btn3 = ???;
    // assign w_btn4 = ???;

    // ---- Step 2: Internal logic in active-high (reads naturally!) ----
    wire w_both_12;     // true when BOTH buttons 1 and 2 pressed
    wire w_either_34;   // true when EITHER button 3 or 4 pressed
    wire w_xor_12;      // true when exactly one of 1/2 pressed
    wire w_not_1;       // true when button 1 is NOT pressed

    // TODO: Implement using the active-high wires
    // assign w_both_12   = ???;
    // assign w_either_34 = ???;
    // assign w_xor_12    = ???;
    // assign w_not_1     = ???;

    // ---- Step 3: Invert outputs at the boundary ----
    // TODO: Drive active-low LEDs from active-high logic
    // assign o_led1 = ???;
    // assign o_led2 = ???;
    // assign o_led3 = ???;
    // assign o_led4 = ???;

endmodule

### Exercise 5 — Stretch: Makefile & XOR Pattern (10 min)

**Goal:** Automate the build flow and experiment with more gate combinations.

- Open `starter/w1d1_ex5_xor_pattern.v` — add creative logic combinations
- The Makefile already supports all exercises; study how it works
- **Challenge:** Can you make all 4 LEDs display a unique pattern based on the 4 buttons?

---

#### 📝 Exercise 5 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex5_xor_pattern.v
// =============================================================================
// Exercise 5 (Stretch): XOR Pattern
// Day 1 · Welcome to Hardware Thinking
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Goal: Create interesting LED patterns using gate combinations.
//
// Challenge: Make each LED respond to a UNIQUE combination of buttons.
// =============================================================================

module xor_pattern (
    input  wire i_switch1,
    input  wire i_switch2,
    input  wire i_switch3,
    input  wire i_switch4,
    output wire o_led1,
    output wire o_led2,
    output wire o_led3,
    output wire o_led4
);

    // Invert at boundary (active-high internally)
    wire w_b1 = ~i_switch1;
    wire w_b2 = ~i_switch2;
    wire w_b3 = ~i_switch3;
    wire w_b4 = ~i_switch4;

    // TODO: Design 4 unique logic functions
    // Ideas:
    //   - XOR of all 4 buttons (odd parity)
    //   - XNOR of pairs
    //   - Majority function (3 of 4 pressed)
    //   - Any creative combination!

    assign o_led1 = 1'b1;  // TODO
    assign o_led2 = 1'b1;  // TODO
    assign o_led3 = 1'b1;  // TODO
    assign o_led4 = 1'b1;  // TODO

endmodule

## Deliverable Checklist

- [ ] Toolchain verified — all tools report version
- [ ] Exercise 1: LED1 lit on board
- [ ] Exercise 2: All 4 buttons control corresponding LEDs
- [ ] Exercise 3: Truth table filled in and verified on hardware
- [ ] Exercise 4: Active-low pattern implemented and working
- [ ] At minimum: Exercise 3 or 4 programmed on board with logic modifications

## Quick Reference

In [ ]:
!make ex1          # Build and program Exercise 1

In [ ]:
!make ex2          # Build and program Exercise 2

In [ ]:
!make ex3          # Build and program Exercise 3

In [ ]:
!make ex4          # Build and program Exercise 4

In [ ]:
!make ex5          # Build and program Exercise 5 (stretch)

In [ ]:
!make ex1_stat     # Show resource usage for Exercise 1

In [ ]:
!make clean        # Remove all build artifacts